# Context Clash: How Contradictory Information Breaks Agents (and How to Fix It)

When an agent gathers information from multiple sources, those sources often disagree. Different pages report different CEO names, different funding amounts, different founding years. This is context clash: contradictory information accumulates in the agent's context window, and its ability to reason degrades, even when the right answer is present and the prompt includes clear priority rules.

In this notebook, we build a company research agent that pulls data from 8 sources and compare two architectures. The first passes raw data through a shared context window, letting contradictions pile up. The second isolates each source in its own context window and only passes summaries upstream. Same model, same subagents, same prompt. The only difference is how information flows, and it changes everything.

## Setup

In [1]:
import sys
from pathlib import Path
import context_clash

from dotenv import load_dotenv
from langsmith import Client

load_dotenv()

# Ensure the notebook directory is on the Python path so local modules are found
notebook_dir = str(Path(".").resolve())
if notebook_dir not in sys.path:
    sys.path.insert(0, notebook_dir)

client = Client()

from context_clash.helpers import (
    result_match_ratio,
    get_metrics_from_experiment,
    display_metrics,
    display_comparison,
)

print("Setup complete")

Setup complete


## The test dataset

We use a synthetic dataset modeling a real-world scenario: gathering company information about a startup called Materialize from 8 different sources. These include the company homepage, Crunchbase, PitchBook, LinkedIn, Glassdoor, news articles, Wikipedia, and Twitter.

Each source contains a mix of correct, outdated, and outright wrong information. The homepage is the most authoritative, but even it has internal tensions. Current-state pages (team, about, careers) reflect reality, while point-in-time pages (blog posts, press releases) may be outdated. The ground truth below represents the correct, current-state answers.

In [2]:
from context_clash.create_dataset import DATASET_NAME, GROUND_TRUTH, create_dataset

create_dataset()

print(f"Ground truth: {GROUND_TRUTH}")
print(f"Dataset: {DATASET_NAME}")

Dataset already exists: context-clash-materialize
Ground truth: {'company_homepage': 'https://materialize.com', 'ceo_name': 'Nate Stewart', 'cto_name': None, 'year_founded': 2019, 'funding_status': 'series_c', 'office_locations': ['New York City'], 'employee_headcount': '51-200', 'total_funding_raised': 98.5}
Dataset: context-clash-materialize


---

## Approach 1: Subagents that share context

In this architecture, 8 specialized subagents are invoked sequentially, each one adding its raw data to a shared message history. An aggregator at the end sees the full accumulated context and produces a structured answer.

Each subagent reads files from its assigned data source and appends the raw results to the conversation. The context grows with each step. By the time the aggregator runs, it has access to everything from all 8 sources and must apply the prompt's source-priority rules (homepage > Crunchbase/PitchBook > LinkedIn/Glassdoor/News > Wikipedia/Twitter) to hundreds of conflicting data points.

In [3]:
from langgraph.graph import END, START, StateGraph
from context_clash.sequential_graph import ResearchState, aggregator_node
from context_clash.sequential_graph import homepage_node, crunchbase_node, pitchbook_node, linkedin_node
from context_clash.sequential_graph import glassdoor_node, news_node, wikipedia_node, twitter_node

builder = StateGraph(ResearchState)

builder.add_node("homepage", homepage_node)
builder.add_node("crunchbase", crunchbase_node)
builder.add_node("pitchbook", pitchbook_node)
builder.add_node("linkedin", linkedin_node)
builder.add_node("glassdoor", glassdoor_node)
builder.add_node("news", news_node)
builder.add_node("wikipedia", wikipedia_node)
builder.add_node("twitter", twitter_node)
builder.add_node("aggregator", aggregator_node)

builder.add_edge(START, "crunchbase")
builder.add_edge("crunchbase", "pitchbook")
builder.add_edge("pitchbook", "linkedin")
builder.add_edge("linkedin", "glassdoor")
builder.add_edge("glassdoor", "news")
builder.add_edge("news", "wikipedia")
builder.add_edge("wikipedia", "homepage")
builder.add_edge("homepage", "twitter")
builder.add_edge("twitter", "aggregator")
builder.add_edge("aggregator", END)

sequential_agent = builder.compile()
sequential_agent.name = "sequential_graph"

print(f"Agent: {sequential_agent.name}")


Agent: sequential_graph


In [4]:
print(f"Running evaluation with 5 repetitions...")

sequential_experiment = client.evaluate(
    lambda x: sequential_agent.invoke(x)["structured_response"].model_dump(),
    data=DATASET_NAME,
    evaluators=[result_match_ratio],
    experiment_prefix="sequential-graph",
    num_repetitions=5,
    max_concurrency=5,
)

sequential_metrics = get_metrics_from_experiment(sequential_experiment)
display_metrics(sequential_metrics, "Sequential Graph Results")

Running evaluation with 5 repetitions...
View the evaluation results for experiment: 'sequential-graph-09249908' at:
https://smith.langchain.com/o/7fbff7fe-62bc-4be3-9d5d-ac6ff3269721/datasets/284faaf5-aad7-486d-82a2-04fbd6a8ea4d/compare?selectedSessions=aad11e38-32cf-451f-a688-62624db1eb5c




0it [00:00, ?it/s]


  Sequential Graph Results
  Metric                              Value
  ----------------------------------------
  Match Ratio                        70.0%
  Latency (p99)                     211.3s
  Avg Tokens                        677,370
  Avg Cost                         $0.7352



### Why the sequential graph fails

The sequential graph struggles because of context clash, the exact failure mode we set out to demonstrate.

Consider the CTO field. The correct answer is `None`: Materialize currently has no CTO. But arriving at that answer requires nuance even within the homepage alone. The team page lists no CTO (current state), while a 2021 blog post announces "Alex Chen as CTO" and a 2023 blog post still references him. The aggregator must recognize that current-state pages (team, about) override point-in-time pages (blog, press).

The aggregator prompt includes strict rules: "current-state homepage pages always win over point-in-time pages." But with hundreds of raw messages in context, the model's reasoning degrades. Early incorrect data persists and influences later conclusions. Even when the right answer is present, the sheer volume of contradictions makes it hard to follow priority rules consistently.

The problem isn't missing data or bad prompting. It's the architecture of the agent.

---

## Approach 2: Subagents-as-tools (context quarantine)

This architecture keep the context for each data source separate: each data source is handled by a subagent running in its own isolated context window. A coordinator agent invokes them as tools and only sees their summaries, never the raw contradictory data.

```
User Query → Coordinator (with 8 tool functions)
               ├─ call_homepage_agent()   → returns summary only
               ├─ call_crunchbase_agent() → returns summary only
               ├─ call_pitchbook_agent()  → returns summary only
               ├─ call_linkedin_agent()   → returns summary only
               ├─ call_glassdoor_agent()  → returns summary only
               ├─ call_news_agent()       → returns summary only
               ├─ call_wikipedia_agent()  → returns summary only
               └─ call_twitter_agent()    → returns summary only
```

Each subagent runs with a fresh context and only sees data from its own source. The coordinator receives concise summaries rather than hundreds of raw contradictory data points, and source attribution is preserved, making the priority rules in the prompt straightforward to apply.

Crucially, the subagent components are identical across both architectures. Both import the exact same subagents from the `subagents/` package with the same prompts, the same tools (`list_files`, `read_file`), and the same `CompanyInfo` output schema. The only difference is how information flows between them.

In [5]:
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy
from context_clash.model import SUBAGENTS_AS_TOOLS_AGGREGATOR_PROMPT, model
from context_clash.subagents.models import CompanyInfo
from context_clash.subagents_as_tools import (
    call_homepage_agent, call_crunchbase_agent, call_pitchbook_agent,
    call_linkedin_agent, call_glassdoor_agent, call_news_agent,
    call_wikipedia_agent, call_twitter_agent,
)

tools_agent = create_agent(
    name="subagents-as-tools",
    model=model,
    tools=[
        call_homepage_agent,
        call_crunchbase_agent,
        call_pitchbook_agent,
        call_linkedin_agent,
        call_glassdoor_agent,
        call_news_agent,
        call_wikipedia_agent,
        call_twitter_agent,
    ],
    system_prompt=SUBAGENTS_AS_TOOLS_AGGREGATOR_PROMPT,
    response_format=ProviderStrategy(schema=CompanyInfo, strict=True),
)

print(f"Agent: {tools_agent.name}")

Agent: subagents-as-tools


In [6]:

print(f"Running evaluation with 5 repetitions...")

tools_experiment = client.evaluate(
    lambda x: tools_agent.invoke(x)["structured_response"].model_dump(),
    data=DATASET_NAME,
    evaluators=[result_match_ratio],
    experiment_prefix="subagents-as-tools",
    num_repetitions=5,
    max_concurrency=5,
)

tools_metrics = get_metrics_from_experiment(tools_experiment)
display_metrics(tools_metrics, "Subagents-as-Tools Results")

Running evaluation with 5 repetitions...
View the evaluation results for experiment: 'subagents-as-tools-e11e7ae3' at:
https://smith.langchain.com/o/7fbff7fe-62bc-4be3-9d5d-ac6ff3269721/datasets/284faaf5-aad7-486d-82a2-04fbd6a8ea4d/compare?selectedSessions=beb70744-53a6-41b6-ad67-3bf1bdf86a31




0it [00:00, ?it/s]


  Subagents-as-Tools Results
  Metric                              Value
  ----------------------------------------
  Match Ratio                       100.0%
  Latency (p99)                      75.0s
  Avg Tokens                         58,496
  Avg Cost                         $0.0860



### Why subagents-as-tools wins

Both architectures use the same model, the same subagents, and the same aggregator prompt with strict source-priority rules. The difference is purely structural. Context isolation means each subagent only sees data from its own source, so contradictions never cross boundaries. Summarization acts as a filter: each subagent distills its source into a concise summary, stripping out noise like conflicting headcounts or fabricated funding rounds before they reach the coordinator. And because the coordinator sees clean, attributed summaries ("Homepage says CEO is Nate Stewart", "Crunchbase says..."), the priority rules in the prompt become straightforward to apply. The sequential graph doesn't fail because of bad prompting. It fails because it forces the model to process all contradictions at once.

## Key takeaways

Context clash is a real failure mode: when contradictory information piles up in a single context window, even strong models with clear priority rules start making errors. The fix here wasn't better prompting (the aggregator prompt is identical in both architectures), it was better architecture. Giving each data source its own isolated context window and only passing summaries upstream lets the coordinator work with clean, attributed information instead of hundreds of conflicting raw messages. The fact that both architectures share the exact same subagent implementations, same prompts, same tools, same output schema, makes context quarantine a low-effort structural change with outsized impact. Running evals with multiple repetitions via LangSmith confirms this isn't luck on a single run.